In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dense, Dropout, Flatten
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import cv2

# Define the model architecture
def create_model(input_shape):
    model = Sequential()
    model.add(Conv2D(32, (3, 3), activation='relu', input_shape=input_shape))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Conv2D(128, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Flatten())
    model.add(Dense(512, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(4, activation='linear'))  # 4 outputs for bounding box coordinates (x, y, w, h)
    return model

# Compile the model
input_shape = (128, 128, 3)
model = create_model(input_shape)
model.compile(optimizer='adam', loss='mean_squared_error')

# Load and preprocess the dataset
def load_data(data_dir):
    images = []
    bboxes = []
    for filename in os.listdir(data_dir):
        if filename.endswith(".jpg"):
            img = cv2.imread(os.path.join(data_dir, filename))
            img = cv2.resize(img, (128, 128))
            img = img.astype('float32') / 255.0  # Normalize the image
            images.append(img)
            # Load corresponding bounding box annotations
            bbox = load_bbox(os.path.join(data_dir, filename.replace(".jpg", ".txt")))
            bboxes.append(bbox)
    return np.array(images), np.array(bboxes)

def load_bbox(bbox_file):
    with open(bbox_file, "r") as f:
        bbox = [float(coord) for coord in f.readline().split()]
    return bbox

data_dir = "C:\\Users\\Model\\datasets"
images, bboxes = load_data(data_dir)
# Train the model
if images.size > 0 and bboxes.size > 0:
    # Train the model
    if images.size > 0 and bboxes.size > 0:
        model.fit(images, bboxes, epochs=10, batch_size=32)
    else:
        print("No data available for training.")
else:
    print("No data available for training.")
model.fit(images, bboxes, epochs=10, batch_size=32)

# Function to detect objects and draw bounding boxes
def detect_objects(image_path, model):
    img = cv2.imread(image_path)
    img_resized = cv2.resize(img, (128, 128))
    img_resized = np.expand_dims(img_resized, axis=0)
    bbox = model.predict(img_resized)[0]
    
    x, y, w, h = bbox
    x = int(x * img.shape[1] / 128)
    y = int(y * img.shape[0] / 128)
    w = int(w * img.shape[1] / 128)
    h = int(h * img.shape[0] / 128)
    
    cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

# Example usage
detect_objects("C:\\Users\\Admin\\Documents\\models\\model-api\\datasets\\Brown Planthopper\\1.jpg", model)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Model\\datasets\\Brown Planthopper\\00000067_jpg.rf.b79c3e157fe7e3159a321bed2f13f837.txt'